<a href="https://colab.research.google.com/github/mbaker21231/MicroII-Sandbox/blob/main/Rubinstein_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rubinstein's model

Rubinstein attacked the bargaining problem from a completely different perspective and built a structural model of bargaining. That is, his idea was to outline a reasonable way of thinking about a variety of bargaining situations as they might unfold.

Advantage: we can add in a lot of bells and whistles, and think about things like breakdown, outside options while bargaining, etc. and how they influence outcomes.

How does it work? Well, we are going to do a lot of so-called "order of" math, so let's get a way together of coping with it.

Here goes:

A lot of what we will do will involve looking at things as quantities become very small. When a value in a polynomial in particular becomes small, we have the result that linear terms dominate all the others, so many things just go to zero.

Consider something like:

$$
P(x) = 1 + ax + bx^2 + cx^3 + dx^4, \quad Q(x)=1+ex+fx^2
$$

Suppose we are interested in a product $P(x)Q(x)$, but we know that eventually we will be looking at small values of $x$, then, we might  write the product as:

$$
P(x)Q(x)= 1 + ax+bx^2 + ex+eax^2+O(x^3)
$$

This notation means that for small enough values of $x$, any term with order 3 or higher will be inconsequential. So, ultimately, we will drop it and say something like:

$$
P(x)Q(x) \approx 1 + ax+bx^2 + ex + eax^2
$$

A convenience is that we can deal with these things very simply in Python...

In [ ]:
from sympy import *

In [ ]:
a, b, c, d, e, f, x = symbols('a b c d e f x')

P = 1 + a*x + b*x**2 + c*x**3
Q = 1 + e*x + f*x**2

In [ ]:
Result = (P*Q).expand()
Result

a*e*x**2 + a*f*x**3 + a*x + b*e*x**3 + b*f*x**4 + b*x**2 + c*e*x**4 + c*f*x**5 + c*x**3 + e*x + f*x**2 + 1

But note in Python, with the sympy module, we can do:

In [ ]:
Result + O(x**3)

1 + f*x**2 + e*x + b*x**2 + a*x + a*e*x**2 + O(x**3)

In [ ]:
(Result + O(x**3)).removeO()

a*e*x**2 + a*x + b*x**2 + e*x + f*x**2 + 1

Why is this useful? We will see it will help us deal with limits as time periods, in particular, get really small. So, here is the simplest Rubinstein-style model for a bargain. Basically, the procedure works as follows:

## Two players and a cake

- Bargaining takes place over time, and time is chopped up into discrete units of length $\delta$.

- The players bargain over the split of a cake of size $\pi$.

- At times $\delta,3\delta,5\delta,\dots$, player one proposes for himself a share $x_1$ of the cake. At times $2\delta,4\delta, 6\delta,\dots$, player two proposes for himself a share $x_2$ of the cake.

- Players have discount factors $\beta_1,\beta_2$. In continuous time, these would be $\beta_1 = e^{-\delta r_i}$, where $r_i$ is the rate at which the player discounts future payoffs.

This last fact allows us to approximate discount factors using a taylor expansion around $r=0$, so:

$$
\beta_1 = 1-\delta r_i + \frac{\delta^2}{2}r_i^2\dots
$$

or:

$$
\beta_1 \approx 1-\delta r_i
$$

# Rubinstein solution

Idea: apply game theoretic concepts to solve for optimal, Subgame Perfect Nash Equilibrium offers. These offer must involve:

1) **Stationarity** - if a player finds an offer optimal at any time, it should be optimal at any other time, and
2) **No delay** - offers made should always be accepted.

The two together make perfect sense...why would a player ever find it optimal to make an offer that is not accepted? Let's label the optimal proposals $x_1^*$ and $x_2^*$. Clearly, we must have:

$$
\pi-x_1^* \geq \beta_2x_2^*, \quad \pi-x_2^* \geq \beta_1x_1^*
$$

The first part of the above means that player one must make an offer that player two will accept. Note the proposals is a share for the player doing the proposing! This offer leaves player two a share of the pie greater than the (discounted value) of the share he or she will propose in the next period.

The second part of the above equation is the related equation for player two. To solve, we observe that player one will satisfy the first part as close to equality as possible, as would player two (so we have a weak subgame perfect nash equilibrium now).

our equations are then:

$$
\pi-x_1^* = \beta_2x_2^*, \quad \pi-x_2^* = \beta_1x_1^*
$$

Let's solve these and bust out some python to do so...

In [ ]:
pi, x_1, x_2, beta_1, beta_2, r_1, r_2, delta = symbols('pi x_1 x_2 beta_1 beta_2 r_1 r_2 delta')

In [ ]:
eq1 = pi - x_1 - beta_2*x_2
eq2 = pi - x_2 - beta_1*x_1
eq1, eq2

(-beta_2*x_2 + pi - x_1, -beta_1*x_1 + pi - x_2)

In [ ]:
solns = solve([eq1, eq2], [x_1, x_2])
x1sta = solns[x_1]
x2sta = solns[x_2]

In [ ]:
x1sta

(beta_2*pi - pi)/(beta_1*beta_2 - 1)

But let's take things a bit further and think about what happens as we let the time period between offers get shorter and shorter. This changes the discount factor, and we will use $\beta = 1-\delta r$, where $\delta$ is the time interval.

In [ ]:
x1 = x1sta.subs({beta_1:1-r_1*delta, beta_2:1-r_2*delta}).expand()
x2 = x2sta.subs({beta_1:1-r_1*delta, beta_2:1-r_2*delta}).expand()
x1, x2

(-delta*pi*r_2/(delta**2*r_1*r_2 - delta*r_1 - delta*r_2),
 -delta*pi*r_1/(delta**2*r_1*r_2 - delta*r_1 - delta*r_2))

In [ ]:
x1 = x1.simplify()
x2 = x2.simplify()
x1, x2

(pi*r_2/(-delta*r_1*r_2 + r_1 + r_2), pi*r_1/(-delta*r_1*r_2 + r_1 + r_2))

In the above, we can let the time period between offers get arbitrarily small, or in other words, go to zero. So:

In [ ]:
x1 = x1.subs({delta:0})
x2 = x2.subs({delta:0})
x1, x2

(pi*r_2/(r_1 + r_2), pi*r_1/(r_1 + r_2))

In [ ]:
x1

pi*r_2/(r_1 + r_2)

Note that if we let the discount factors be equal, we get the Rubinstein solution, or better yet, a foundation for the generalized bargaining solution.

# More nuanced models

So, the neat thing about this setup is that we can expand upon it in various ways. So, for example, suppose that there is an impasse option. In particular, suppose that we have each person get some flow payoff whenever there is a delay.

That is, there is an a chance that the cake explodes while we are arguing.

Now, we have equations:

$$
\pi-x_1^* = (1-p)\beta_2x_2^*+ pd_2 , \quad \pi-x_2^* = (1-p)\beta_1x_1^* + pd_1
$$

Our equations become:

In [ ]:
pi, x_1, x_2, beta_1, beta_2, r_1, r_2, r, delta = symbols('pi x_1 x_2 beta_1 beta_2 r_1 r_2 r delta')
p, d_1, d_2, lam = symbols('p d_1 d_2 lambda')

init_printing()

eq1 = pi - x_1 - ((1-p)*beta_2*x_2 + p*d_2)
eq2 = pi - x_2 - ((1-p)*beta_1*x_1 + p*d_1)
eq1, eq2

(-β₂⋅x₂⋅(1 - p) - d₂⋅p + π - x₁, -β₁⋅x₁⋅(1 - p) - d₁⋅p + π - x₂)

In [ ]:
solns = solve([eq1, eq2], [x_1, x_2])

In [ ]:
solns

⎧           2                                                  2               ↪
⎪    β₂⋅d₁⋅p  - β₂⋅d₁⋅p - β₂⋅p⋅π + β₂⋅π + d₂⋅p - π      β₁⋅d₂⋅p  - β₁⋅d₂⋅p - β ↪
⎨x₁: ─────────────────────────────────────────────, x₂: ────────────────────── ↪
⎪                 2                                                  2         ↪
⎩          β₁⋅β₂⋅p  - 2⋅β₁⋅β₂⋅p + β₁⋅β₂ - 1                   β₁⋅β₂⋅p  - 2⋅β₁⋅ ↪

↪                        ⎫
↪ ₁⋅p⋅π + β₁⋅π + d₁⋅p - π⎪
↪ ───────────────────────⎬
↪                        ⎪
↪ β₂⋅p + β₁⋅β₂ - 1       ⎭

These are a little more complicated. But we can plug in our expression for discount factors, and _also model the probability of a breakdown as driven by a poisson process_, which implies that the chances of a breakdown occurring are, in a small interval of time:

$$
p=\delta \lambda,\quad 1-p = 1-\delta \lambda
$$

In [ ]:
x_1 = solns[x_1].subs({beta_2:1-r_2*delta, beta_1:1-r_1*delta, p:lam*delta})
x_1.simplify()

      2  2                                                                     ↪
- d₁⋅δ ⋅λ ⋅(δ⋅r₂ - 1) + d₁⋅δ⋅λ⋅(δ⋅r₂ - 1) + d₂⋅δ⋅λ + δ⋅λ⋅π⋅(δ⋅r₂ - 1) - π⋅(δ⋅r ↪
────────────────────────────────────────────────────────────────────────────── ↪
  2  2                                                                         ↪
 δ ⋅λ ⋅(δ⋅r₁ - 1)⋅(δ⋅r₂ - 1) - 2⋅δ⋅λ⋅(δ⋅r₁ - 1)⋅(δ⋅r₂ - 1) + (δ⋅r₁ - 1)⋅(δ⋅r₂  ↪

↪           
↪ ₂ - 1) - π
↪ ──────────
↪           
↪ - 1) - 1  

Simplification doesn't really help all that much here, so we need to move along and do things with a bit more nuance. Let's use a little order mathematics to get things going. We first separate the numerator and denominator, as these things don't work all that well through numerators:

In [ ]:
x_1n, x_1d = fraction(x_1)
x_1n = x_1n.expand()
x_1d = x_1d.expand()
x_1n, x_1d

⎛      3  2          2  2       2                           2                  ↪
⎝- d₁⋅δ ⋅λ ⋅r₂ + d₁⋅δ ⋅λ  + d₁⋅δ ⋅λ⋅r₂ - d₁⋅δ⋅λ + d₂⋅δ⋅λ + δ ⋅λ⋅π⋅r₂ - δ⋅λ⋅π - ↪

↪           4  2          3  2       3  2         3            2  2      2     ↪
↪  δ⋅π⋅r₂, δ ⋅λ ⋅r₁⋅r₂ - δ ⋅λ ⋅r₁ - δ ⋅λ ⋅r₂ - 2⋅δ ⋅λ⋅r₁⋅r₂ + δ ⋅λ  + 2⋅δ ⋅λ⋅r ↪

↪        2         2                            ⎞
↪ ₁ + 2⋅δ ⋅λ⋅r₂ + δ ⋅r₁⋅r₂ - 2⋅δ⋅λ - δ⋅r₁ - δ⋅r₂⎠

Now, we can apply order mathematics and reconstitute the entire expression as follows:

In [ ]:
x_1sta = ((x_1n + O(delta**2)).removeO() / (x_1d + O(delta**2)).removeO()).simplify()
x_1sta

d₁⋅λ - d₂⋅λ + λ⋅π + π⋅r₂
────────────────────────
     2⋅λ + r₁ + r₂      

So, what if we use equal discount factors now? We have:

In [ ]:
x_1sta.subs({r_2:r, r_1:r}).collect(pi)

d₁⋅λ - d₂⋅λ + π⋅(λ + r)
───────────────────────
       2⋅λ + 2⋅r       

Note what happens if we assume agents are impatient:

In [ ]:
x_1sta.subs({r_1:0, r_2:0}).expand()

d₁   d₂   π
── - ── + ─
2    2    2

A further nuance - is breakdown different from impasse? When we are at an impasse, we will bargain again next period. We might model this as the two equations:

$$
\pi-x_1^* =u_2+ \beta_2x_2^*, \quad \pi-x_2^* = u_1+\beta_1x_1^*
$$

In this case, the player collects utility $u$ while waiting for the next round of bargaining. Of course, we would really just attack this the same as before, but now allow $\delta u$ to be the flow utility. We have:

In [ ]:
pi, x_1, x_2, beta_1, beta_2, r_1, r_2, r, u_1, u_2, delta = symbols('pi x_1 x_2 beta_1 beta_2 r_1 r_2 r u_1 u_2 delta')
p, d_1, d_2, lam = symbols('p d_1 d_2 lambda')

In [ ]:
eq1 = pi - x_1 - (beta_2*x_2 + u_2)
eq2 = pi - x_2 - (beta_1*x_1 + u_1)
solns = solve([eq1, eq2], [x_1, x_2])
solns


⎧    β₂⋅π - β₂⋅u₁ - π + u₂      β₁⋅π - β₁⋅u₂ - π + u₁⎫
⎨x₁: ─────────────────────, x₂: ─────────────────────⎬
⎩          β₁⋅β₂ - 1                  β₁⋅β₂ - 1      ⎭

In [ ]:
x_1 = solns[x_1].subs({beta_2:1-r_2*delta, beta_1:1-r_1*delta, u_1:u_1*delta, u_2:u_2*delta})
x_1.simplify()

δ⋅r₂⋅u₁ - π⋅r₂ - u₁ + u₂
────────────────────────
   δ⋅r₁⋅r₂ - r₁ - r₂    

We don't really need to apply all the tricks we did before, but now can just let $\delta$ go to zero to get:

$$
x_1^*=\frac{r_2\pi}{r_2+r_1}+\frac{u_1-u_2}{r_1+r_2}
$$

Note the interpretation here...and how discount factors interact with the solution